In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os
from pypdf import PdfReader

In [2]:
load_dotenv(override=True)

True

In [3]:
def extract_pdf_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text=""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text+=page_text+"\n"
    
    return text


In [4]:
document_text = extract_pdf_text("me/Mahendra_CV.pdf")

In [5]:
def chunk_text(text,max_words=1200):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        current_chunk.append(word)
        if len(current_chunk) >= max_words:
            chunks.append(" ".join(current_chunk))
            current_chunk=[]
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [9]:
client = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)

In [12]:
def analyze_chunk(chunk_text: str) -> str:
    prompt = f"""Your are a Document Analyzer Agent. 
    Instructions: 
    - Extract key ideas and facts
    - Use bullet points
    - Do not summarize or rewrite
    - Do not add opinions
    
    Document: {chunk_text}
    """

    response = client.chat.completions.create(
        model="openai.o4-mini",
        messages=[{"role":"system", "content": "You are an expert document analyst."},{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [13]:
chunks = chunk_text(document_text)

all_key_points = []
for chunk in chunks:
    key_points = analyze_chunk(chunk)
    all_key_points.append(key_points)

combined_key_points = "\n".join(all_key_points)

In [14]:
print("===Key Points===")
print(combined_key_points)

===Key Points===
• Name: Mahendra Girase  
• Designation: Analyst / Software Engineer  
• Birth Date: 02/04/2004  
• Nationality: Indian  
• Marital Status: Single  
• Gender: Male  
• Passport Details: NA  
• Country: India  

• Current Role: Analyst, A4 BU PBS  
• Date of Joining: 26th June 2025  
• Service Line: SAP PBS  

• Profile:  
  – Driven software engineer with expertise in backend development, integration technologies, and AI  
  – Focus on continuous learning, problem-solving, and team contribution  

• Programming Languages: Java, JavaScript, Python, C, SQL, ABAP  
• Backend & Integration Technologies: Spring Boot, Spring MVC, Spring Data JPA, Spring Security, Spring AI, SAP CPI, SAP PI/PO, SAP ABAP, Groovy, Microservices  
• Databases: MySQL, PostgreSQL, MongoDB  
• Frameworks & Libraries: Spring Framework, Hibernate, React, Node.js, Express.js  
• Tools & Platforms: Git, GitHub, Maven, Postman, Docker, Cloud Deployment  
• AI/ML Technologies: Generative AI (LLMs), Agent

In [17]:
def summarize_key_points(key_points: str) -> str:
    prompt = f"""You are a Summarization Agent.
    Instructions:
    -Write a clear and concise summary
    -Use simple professional language
    -Maximum 150 words
    -No bullet repetition
    
    key Points:{key_points}"""

    response = client.chat.completions.create(
        model="openai.o4-mini",
        messages=[{"role":"system", "content": "You are an expert technical writer."},{
            "role":"user", "content":prompt
        }]
    )

    return response.choices[0].message.content


In [18]:
final_summary = summarize_key_points(combined_key_points)

print("===Final Summary===")
print(final_summary)

===Final Summary===
Mahendra Girase is an analyst and software engineer at Capgemini’s A4 BU PBS, joining on June 26, 2025. With a B.Tech from Vivekanand Education Society Institute of Technology (CGPA 9.28), he specializes in backend development, integration technologies and AI. Proficient in Java, JavaScript, Python, C, SQL and ABAP, he works with Spring Boot, SAP CPI/PI-PO, microservices, Hibernate, React and Node.js. Mahendra has designed iFlows using SOAP, HTTP and JMS adapters, developed REST APIs secured with JWT, and built AI-enabled applications with Spring AI and Docker. He led an Agentic AI project using Python, OpenAI API and CrewAI to create autonomous multi-agent workflows. His certifications include SAP CPI, Java Spring Boot, Generative AI and leadership training. Based in India, he is committed to continuous learning, problem-solving and team collaboration.
